# Hybrid ALNS for 1D Bin Packing: concise first-principles notes

## 1) Problem and objective
Given item sizes $s_i \in \mathbb{Z}_{>0}$ and bin capacity $C$, assign each item to exactly one bin so that each bin load is at most $C$ and the number of used bins is minimized.

A solution can be represented by bins $B_1,\dots,B_m$ with constraints:
- $\bigcup_j B_j = \{1,\dots,n\}$ and bins are disjoint.
- $\sum_{i\in B_j} s_i \le C$ for each $j$.
- Objective: minimize $m$.

Lower bound: $\left\lceil \frac{\sum_i s_i}{C} \right\rceil$. This is necessary but not generally sufficient.

## 2) Why local search gets stuck
Neighborhood search applies small edits (move/swap/repack limited items). For NP-hard combinatorial structure, many local minima appear: every small edit worsens or is infeasible, even though a better global arrangement exists.

## 3) ALNS idea from first principles
**Large Neighborhood Search (LNS)** escapes local minima by making large edits in two phases:
1. **Destroy:** remove a subset of assignments (partial unassignment).
2. **Repair:** reinsert removed items with a constructive heuristic.

**Adaptive LNS (ALNS)** keeps multiple destroy/repair operators and updates their selection probabilities using observed performance.

This implementation uses:
- Destroy operators: random bins, worst bins, related items.
- Repair: learned bin-choice policy (classification scores over feasible bins).
- Adaptive operator selection: Thompson Sampling bandit.
- Acceptance: Simulated Annealing criterion.

## 4) Acceptance and exploration
Let cost be number of bins. For candidate $x'$ from current $x$, define $\Delta = f(x')-f(x)$.
- If $\Delta \le 0$: accept.
- Else accept with probability $\exp(-\Delta/T)$.

Temperature $T$ decreases geometrically, so search is exploratory early and greedy later.

## 5) Thompson Sampling for operator choice
Treat each destroy operator as a bandit arm with unknown success probability.
- Maintain Beta posterior per arm: $\text{Beta}(\alpha_k,\beta_k)$.
- Sample one value from each posterior, choose max sample.
- Update chosen arm with reward 1 (useful accepted improvement) or 0.

This balances exploration/exploitation with principled uncertainty handling.

## 6) Learned repair model
Repair chooses among feasible bins for each displaced item.
For each (item, feasible-bin) pair, compute features (item size, bin load, residual capacity, slack-after-placement, rank/context features). A logistic model outputs a score; choose the best feasible bin. If none feasible, open a new bin.

Training data is generated offline by replaying BFD decisions and labeling:
- Positive: bin chosen by BFD.
- Negative: other feasible bins.

So the model imitates a strong heuristic while ALNS destroy phases create opportunities BFD alone cannot reach.

## 7) Workflow
1. Train repair model once and save `repair_model.pkl`.
2. Run benchmark with hybrid solver and pass model path via `--method-args model_path=...`.
3. Read summary output.


In [ ]:
!python 5_hybrid_ml_metaheuristics/hybrid_alns/train_repair_model.py \
  --instances 3000 \
  --n-min 50 \
  --n-max 200 \
  --max-negatives 5 \
  --seed 0 \
  --output 5_hybrid_ml_metaheuristics/hybrid_alns/repair_model.pkl

In [ ]:
!python benchmark.py \
  --solver 5_hybrid_ml_metaheuristics/hybrid_alns/solver.py \
  --method hybrid_alns \
  --method-args model_path=5_hybrid_ml_metaheuristics/hybrid_alns/repair_model.pkl,max_iterations=5000 \
  --datasets falkenauer-t \
  --limit 5